# LVC Voice 6 — Build loader bảo mật (.so)

**Chỉ bạn chạy.** Notebook này Cython hoá `remote/voice_loader.py` thành
`voice_loader.*.so` (khớp Python + nền tảng của Colab), rồi in checksum. Tải
file `.so` về, đưa lên Release `loader-linux` với tên `voice_loader-cp313-linux-x86_64.so` — notebook khách
kéo từ đó.

Loader .so chứa kill-switch; giao khách dạng binary thì khó gỡ hơn nhiều so với
giao mã nguồn.


In [ ]:
# ── Build loader -> .so ───────────────────────────────────────────────
import os, sys, subprocess, hashlib
from pathlib import Path

REPO = "https://github.com/GrayPham/submodulevoice.git"
BRANCH = "master"
APP = Path("/content/app")
PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"Python Colab: {sys.version.split()[0]} (tag {PYTAG})")

def sh(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         start_new_session=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"lỗi ({p.returncode}): {cmd}")

if APP.exists():
    sh(f"git -C {APP} fetch --depth 1 origin {BRANCH} && git -C {APP} reset --hard origin/{BRANCH}")
else:
    sh(f"git clone --depth 1 -b {BRANCH} {REPO} {APP}")
sh("pip install -q cython httpx cryptography")

# Cython hoá voice_loader.py (đứng ở gốc app để --inplace đặt .so đúng chỗ).
from Cython.Build import cythonize
from setuptools import Extension
from setuptools.dist import Distribution

os.chdir(APP)
rel = "remote/voice_loader.py"
ext = Extension("remote.voice_loader", [rel])
dist = Distribution({"ext_modules": cythonize([ext], language_level=3,
    compiler_directives={"emit_code_comments": False}, quiet=True)})
dist.script_args = ["build_ext", "--inplace", "--build-temp", "/tmp/cyb"]
dist.parse_command_line(); dist.run_commands()

so = list((APP / "remote").glob("voice_loader*.so"))
assert so, "không sinh được .so"
src = so[0]
# Xoá nguồn + .c để không phát tán kèm.
for junk in (APP / rel, APP / "remote" / "voice_loader.c"):
    if junk.exists():
        junk.unlink()

# Đổi tên chuẩn hoá để notebook khách tải đúng một tên cố định.
out = Path("/content") / "voice_loader-cp313-linux-x86_64.so"
out.write_bytes(src.read_bytes())
digest = hashlib.sha256(out.read_bytes()).hexdigest()
(Path("/content") / ("voice_loader-cp313-linux-x86_64.so" + ".sha256")).write_text(f"{digest}  {out.name}\n")

print(f"\n=== XONG ===")
print(f"Loader : {out}  ({out.stat().st_size/1024:.0f} KB)")
print(f"SHA-256: {digest}")
print(f"\nTải {out.name} + .sha256 về, đưa lên Release 'loader-linux'.")
print(f"(PYTAG={PYTAG} — phải khớp Python mà notebook khách chạy.)")
